# Smoke Test Verifikasi Realisasi Indikator RPJMN 2025-2029

Membandingkan **2 model** x **2 search backend** pada **10 baris** gold standard.

| | |
|---|---|
| Model | `watsonx-qwen3-30b-a3b-instruct-2507`, `gemma-4-26B-A4B-it` |
| Backend | DuckDuckGo (gratis), Tavily (1.000 kredit/bulan gratis) |
| Sample | 10 baris stratified: **7 TRUE + 3 FALSE** |
| Total | 4 kombinasi x 10 baris = **40 task** |


### Prasyarat
```bash
pip install openai pandas python-dotenv ddgs tavily-python
```
`.env` di root project:
```
MODELHUB_LLM_API_KEY=...
MODELHUB_LLM_URL=https://api-modelhub.aiplayground.id
TAVILY_API_KEY=...        # opsional; tanpa ini backend Tavily dilewati
```

## 1. Setup

In [1]:
import os, re, json, time, hashlib, warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

warnings.filterwarnings("ignore")
load_dotenv()

API_KEY  = os.getenv("MODELHUB_LLM_API_KEY")
BASE_URL = os.getenv("MODELHUB_LLM_URL", "https://api-modelhub.aiplayground.id").rstrip("/")
if not BASE_URL.endswith("/v1"):
    BASE_URL = f"{BASE_URL}/v1"
if not API_KEY:
    raise RuntimeError("MODELHUB_LLM_API_KEY belum diset di .env")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

DATA_RAW = Path("../data/raw")
DATA_OUT = Path("../data/output")
CACHE_DIR = Path("../data/cache")
DATA_OUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

print("Base URL :", BASE_URL)
print("Tavily   :", "ADA" if os.getenv("TAVILY_API_KEY") else "TIDAK ADA (backend tavily akan dilewati)")
print("Run ID   :", RUN_TS)

Base URL : https://api-modelhub.aiplayground.id/v1
Tavily   : ADA
Run ID   : 20260917_131242


## 2. Data (sample stratified)

Ambil **7 baris TRUE pertama** + **3 baris FALSE pertama**. Deterministik, jadi hasil antar-run bisa dibandingkan.

In [3]:
CSV_PATH = DATA_RAW / "gold_standard_sample.csv"
gold = pd.read_csv(CSV_PATH)
gold.columns = [c.strip() for c in gold.columns]
gold = gold.loc[:, ~gold.columns.str.startswith("Unnamed")]

COL_REAL = "realisasi (TRUE/FALSE)"
gold["_gold"] = gold[COL_REAL].astype(str).str.strip().str.upper().eq("TRUE")

print(f"Total baris : {len(gold)}")
print(f"TRUE / FALSE: {gold['_gold'].sum()} / {(~gold['_gold']).sum()}")

idx_true  = list(gold[gold["_gold"]].index[:7])
idx_false = list(gold[~gold["_gold"]].index[:3])
SAMPLE = gold.loc[idx_true + idx_false].reset_index(drop=True)

print(f"\nSample: {len(SAMPLE)} baris ({SAMPLE['_gold'].sum()} TRUE, {(~SAMPLE['_gold']).sum()} FALSE)")
print(f"Baseline 'asal TRUE' = {SAMPLE['_gold'].mean():.0%}\n")

for _, r in SAMPLE.iterrows():
    print(f"  no={r['no']:<4} [{'TRUE ' if r['_gold'] else 'FALSE'}] {r['sektor'][:14]:16s} | {str(r['sub_indikator'])[:62]}")

Total baris : 292
TRUE / FALSE: 286 / 6

Sample: 10 baris (7 TRUE, 3 FALSE)
Baseline 'asal TRUE' = 70%

  no=1    [TRUE ] Kesehatan        | KP 02.12.08 - Kabupaten/kota yang mendeklarasikan 5 Pilar STBM
  no=2    [TRUE ] Pertahanan       | KP 02.09.03 - Persentase kebijakan di bidang kerja sama pemban
  no=3    [TRUE ] Industri         | KP 02.09.04 - Persentase keberhasilan promosi, pembentukan nor
  no=4    [TRUE ] Sosial           | KP 07.16.02 - Indeks Diplomasi Perlindungan WNI di Luar Negeri
  no=5    [TRUE ] Pertahanan       | KP 02.01.01 - Persentase pemenuhan alutsista
  no=6    [TRUE ] Pangan           | KP 02.10.13 - Persentase wilayah terkendali dari Hama Penyakit
  no=7    [TRUE ] Infrastruktur    | PP 02.12 - Efisiensi pemanfaatan air irigasi
  no=11   [FALSE] Pangan           | KP 02.10.17 - Produksi vanili
  no=12   [FALSE] Pangan           | KP 02.10.18 - Persentase produksi komoditas pertanian organik 
  no=44   [FALSE] Sosial           | KP 06.03.02 - Jumlah penerim

## 3. Search backend

Dua backend dengan **signature identik**, jadi bisa ditukar tanpa mengubah kode lain.

Cache disimpan ke disk (`../data/cache/`) supaya:
- query yang sama dari model berbeda hanya dibayar sekali
- restart kernel tidak menghanguskan hasil yang sudah dibayar

In [10]:
SEARCH_COUNTER = {"ddg": 0, "tavily": 0, "cache_hit": 0}

def _cache_path(backend):
    return CACHE_DIR / f"search_cache_{backend}.json"


def _load_cache(backend):
    p = _cache_path(backend)
    if p.exists():
        try:
            return json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}

_CACHE = {"ddg": _load_cache("ddg"), "tavily": _load_cache("tavily")}

def _save_cache(backend):
    _cache_path(backend).write_text(
        json.dumps(_CACHE[backend], ensure_ascii=False),
        encoding="utf-8",
    )

def _key(query, situs):
    return hashlib.md5(f"{query}||{situs or ''}".encode()).hexdigest()


def _ddg(query, n=6):
    from ddgs import DDGS
    for attempt in range(3):
        try:
            with DDGS() as d:
                hits = list(d.text(query, region="id-id", max_results=n))
            return [{"judul": h.get("title", ""), "url": h.get("href", ""),
                     "cuplikan": (h.get("body") or "")[:400]} for h in hits]
        except Exception as e:
            if attempt == 2:
                return [{"error": f"{type(e).__name__}: {str(e)[:120]}"}]
            time.sleep(4 * (attempt + 1))


def _tavily(query, n=6):
    from tavily import TavilyClient
    tv = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    try:
        res = tv.search(query, max_results=n, search_depth="basic")
        return [{"judul": h.get("title", ""), "url": h.get("url", ""),
                 "cuplikan": (h.get("content") or "")[:400]} for h in res.get("results", [])]
    except Exception as e:
        return [{"error": f"{type(e).__name__}: {str(e)[:120]}"}]


def cari_web(query, situs=None, backend="ddg"):
    """Signature yang dilihat model. Mengembalikan list hasil pencarian."""
    q = f"site:{situs} {query}" if situs else query
    k = _key(q, None)

    if k in _CACHE[backend]:
        SEARCH_COUNTER["cache_hit"] += 1
        return _CACHE[backend][k]

    hasil = _ddg(q) if backend == "ddg" else _tavily(q)
    SEARCH_COUNTER[backend] += 1

    _CACHE[backend][k] = hasil
    if SEARCH_COUNTER[backend] % 10 == 0:
        _save_cache(backend)
    return hasil


# --- cek cepat kedua backend ---
for bk in ["ddg", "tavily"]:
    if bk == "tavily" and not os.getenv("TAVILY_API_KEY"):
        print(f"[{bk:6s}] dilewati (tidak ada API key)")
        continue
    h = cari_web("laporan kinerja 2025", situs="kemenperin.go.id", backend=bk)
    if h and "error" in h[0]:
        print(f"[{bk:6s}] GAGAL: {h[0]['error']}")
    else:
        print(f"[{bk:6s}] OK, {len(h)} hasil | contoh: {h[0]['judul'][:60]}")

h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof


[ddg   ] GAGAL: DDGSException: RequestError: RequestError('error sending request for url (https://search.yahoo.com/search;_ylt=crhI_C0hlUrmkPMI3mpeArTR
[tavily] OK, 6 hasil | contoh: Penyampaian Laporan Kinerja Tahun 2025


## 4. Prompt

In [ ]:
SYSTEM_PROMPT = """Kamu verifikator realisasi indikator RPJMN 2025-2029 Indonesia.
Tentukan apakah indikator ini SUDAH DILAKSANAKAN pada TAHUN 2025, berdasar publikasi RESMI.

Prioritas sumber (cari berurutan):
1. Situs K/L pengampu: laporan kinerja/LKj/LAKIP 2025, statistik resmi, siaran pers, PPID
2. Bappenas: e-Monev, RKP, Renstra K/L
3. Portal pemerintah lain: BPS, Setkab
4. Media nasional yang mengutip pejabat/dokumen resmi

TOLAK sebagai bukti: blog, opini, analisis konsultan, media sosial pribadi.

Label:
- true  = ada publikasi resmi yang menunjukkan kegiatan ini BERJALAN pada 2025.
          Belum selesai atau belum capai target tetap true.
- false = tidak ada bukti pelaksanaan 2025. Termasuk bila yang ada hanya
          rencana, target, anggaran, atau rapat; atau buktinya dari tahun lain.

Pakai tool cari_web maksimal 4 kali. Bila tidak ketemu bukti, jawab false.
JANGAN mengarang URL: source_url harus berasal dari hasil pencarian.

Balas HANYA JSON, tanpa markdown:
{"realisasi": true, "implementation_summary": "Program berjalan, ditandai dengan ...", "source_url": "https://...", "confidence": "tinggi"}
confidence: tinggi | sedang | rendah"""


def user_prompt(row):
    return (
        f"Sektor: {row['sektor']}\n"
        f"K/L: {row['kl']}\n"
        f"Indikator: {row['sub_indikator']}\n"
        f"Target 2025: {row['target_2025']} {row['satuan']} (baseline 2024: {row['baseline_2024']})\n"
        f"Kata kunci: {row['keywords']}"
    )


TOOLS = [{
    "type": "function",
    "function": {
        "name": "cari_web",
        "description": "Cari di web. Isi 'situs' untuk membatasi ke domain resmi, mis. kemenperin.go.id",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Kata kunci pencarian"},
                "situs": {"type": "string", "description": "Domain, opsional"},
            },
            "required": ["query"],
        },
    },
}]

print(SYSTEM_PROMPT)
print("\n--- contoh user prompt ---")
print(user_prompt(SAMPLE.iloc[0]))

Kamu verifikator realisasi indikator RPJMN 2025-2029 Indonesia.
Tentukan apakah indikator ini SUDAH DILAKSANAKAN pada TAHUN 2025, berdasar publikasi RESMI.

Prioritas sumber (cari berurutan):
1. Situs K/L pengampu: laporan kinerja/LKj/LAKIP 2025, statistik resmi, siaran pers, PPID
2. Bappenas: e-Monev, RKP, Renstra K/L
3. Portal pemerintah lain: BPS, Setkab
4. Media nasional yang mengutip pejabat/dokumen resmi

TOLAK sebagai bukti: blog, opini, analisis konsultan, media sosial pribadi.

Label:
- true  = ada publikasi resmi yang menunjukkan kegiatan ini BERJALAN pada 2025.
          Belum selesai atau belum capai target tetap true.
- false = tidak ada bukti pelaksanaan 2025. Termasuk bila yang ada hanya
          rencana, target, anggaran, atau rapat; atau buktinya dari tahun lain.

Pakai tool cari_web maksimal 4 kali. Bila tidak ketemu bukti, jawab false.
JANGAN mengarang URL: source_url harus berasal dari hasil pencarian.

Balas HANYA JSON, tanpa markdown:
{"realisasi": true, "impleme

## 5. Tool-calling loop

Inti pipeline. Model meminta pencarian, notebook menjalankannya, hasilnya dikirim balik,
diulang sampai model siap menjawab.

In [ ]:
def parse_json(txt):
    if not txt:
        return None
    t = re.sub(r"```(?:json)?", "", txt).strip()
    m = re.search(r"\{.*\}", t, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None


def verifikasi(row, model, backend, max_rounds=6, max_tokens=1200, temperature=0.1):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt(row)},
    ]
    seen_urls, queries = [], []
    n_search, in_tok, out_tok = 0, 0, 0
    t0 = time.perf_counter()

    for _ in range(max_rounds):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, tools=TOOLS, tool_choice="auto",
                temperature=temperature, max_tokens=max_tokens, timeout=180,
            )
        except Exception as e:
            return {"status": "FAIL", "error": f"{type(e).__name__}: {str(e)[:160]}",
                    "latency_s": round(time.perf_counter() - t0, 2),
                    "n_search": n_search, "in_tok": in_tok, "out_tok": out_tok,
                    "queries": queries, "seen_urls": seen_urls}

        u = resp.usage
        in_tok += getattr(u, "prompt_tokens", 0) or 0
        out_tok += getattr(u, "completion_tokens", 0) or 0
        msg = resp.choices[0].message

        if msg.tool_calls:
            messages.append({
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [{"id": tc.id, "type": "function",
                                "function": {"name": tc.function.name,
                                             "arguments": tc.function.arguments}}
                               for tc in msg.tool_calls],
            })
            for tc in msg.tool_calls:
                try:
                    args = json.loads(tc.function.arguments or "{}")
                except Exception:
                    args = {}
                q, situs = args.get("query", ""), args.get("situs")
                queries.append(f"{q}" + (f" [site:{situs}]" if situs else ""))
                hasil = cari_web(q, situs, backend=backend) if q else []
                n_search += 1
                seen_urls += [h.get("url", "") for h in hasil if h.get("url")]
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(hasil, ensure_ascii=False)[:6000]})
            continue

        txt = (msg.content or "").strip()
        if not txt:
            ak = msg.model_dump().get("provider_specific_fields") or {}
            txt = (ak.get("reasoning_content") or "").strip()

        data = parse_json(txt)
        elapsed = round(time.perf_counter() - t0, 2)

        if data is None:
            return {"status": "PARSE_FAIL", "raw": txt[:400], "error": "",
                    "latency_s": elapsed, "n_search": n_search,
                    "in_tok": in_tok, "out_tok": out_tok,
                    "queries": queries, "seen_urls": seen_urls}

        src = str(data.get("source_url") or "").strip()
        return {
            "status": "OK", "error": "",
            "pred": bool(data.get("realisasi")),
            "summary": str(data.get("implementation_summary") or "").strip(),
            "source_url": src,
            "confidence": str(data.get("confidence") or "").strip(),
            "url_dari_search": bool(src) and any(src.rstrip("/") == u.rstrip("/") for u in seen_urls),
            "latency_s": elapsed, "n_search": n_search,
            "in_tok": in_tok, "out_tok": out_tok,
            "queries": queries, "seen_urls": seen_urls,
        }

    return {"status": "MAX_ROUNDS", "error": "", "latency_s": round(time.perf_counter() - t0, 2),
            "n_search": n_search, "in_tok": in_tok, "out_tok": out_tok,
            "queries": queries, "seen_urls": seen_urls}

## 6. Uji satu baris dulu

In [15]:
row = SAMPLE.iloc[0]
print("INDIKATOR:", row["sub_indikator"])
print("K/L      :", row["kl"])
print("GOLD     :", "TRUE" if row["_gold"] else "FALSE")
print("=" * 78)

hasil = verifikasi(row, "watsonx-qwen3-30b-a3b-instruct-2507", "tavily")

print("STATUS   :", hasil["status"], f"| {hasil['latency_s']}s | {hasil['n_search']} pencarian")
print("TOKEN    :", f"in={hasil['in_tok']} out={hasil['out_tok']}")
print("\nQUERY:")
for q in hasil["queries"]:
    print("  -", q)
if hasil["status"] == "OK":
    print("\nPRED     :", hasil["pred"], "| confidence:", hasil["confidence"])
    print("URL      :", hasil["source_url"])
    print("URL asli?:", hasil["url_dari_search"])
    print("SUMMARY  :", hasil["summary"])
else:
    print("\nDETAIL:", hasil.get("error") or hasil.get("raw"))

INDIKATOR: KP 02.12.08 - Kabupaten/kota yang mendeklarasikan 5 Pilar STBM
K/L      : KEMENTERIAN KESEHATAN
GOLD     : TRUE
STATUS   : OK | 5.58s | 4 pencarian
TOKEN    : in=5241 out=355

QUERY:
  - STBM 5 pilar deklarasi kabupaten kota 2025 Kemenkes [site:kemenkes.go.id]
  - LAKIP Kemenkes 2025 STBM 5 pilar [site:kemenkes.go.id]
  - program sanitasi total 2025 Kemenkes deklarasi 5 pilar [site:bappenas.go.id]
  - BPS data sanitasi desa 2025 STBM 5 pilar [site:bps.go.id]

PRED     : False | confidence: tinggi
URL      : 
URL asli?: False
SUMMARY  : Tidak ditemukan publikasi resmi dari Kemenkes, Bappenas, BPS, atau portal pemerintah lain yang menunjukkan pelaksanaan deklarasi 5 Pilar STBM oleh 30 kabupaten/kota pada tahun 2025. Hasil pencarian hanya menunjukkan informasi umum tentang STBM, video edukasi, atau data dari tahun sebelumnya. Tidak ada laporan kinerja, LAKIP, atau siaran pers resmi yang menyebutkan capaian indikator tersebut di tahun 2025.


## 7. Jalankan 4 kombinasi

In [ ]:
MODELS = [
    "watsonx-qwen3-30b-a3b-instruct-2507",
    "gemma-4-26B-A4B-it",
]
BACKENDS = ["tavily"]

print(f"Kombinasi: {len(MODELS)} model x {len(BACKENDS)} backend x {len(SAMPLE)} baris "
      f"= {len(MODELS)*len(BACKENDS)*len(SAMPLE)} task\n")

rows = []
for backend in BACKENDS:
    for model in MODELS:
        tag = f"{model.split('/')[-1][:24]} | {backend}"
        print(f"\n{'='*78}\n{tag}\n{'='*78}")
        for i, r in SAMPLE.iterrows():
            h = verifikasi(r, model, backend)
            ok = h["status"] == "OK"
            benar = (h.get("pred") == r["_gold"]) if ok else None
            mark = ("BENAR" if benar else "SALAH") if ok else h["status"]
            print(f"  [{i+1:2d}/{len(SAMPLE)}] no={r['no']:<4} gold={'T' if r['_gold'] else 'F'} "
                  f"pred={'T' if h.get('pred') else 'F' if ok else '-'} "
                  f"{mark:10s} {h['n_search']}x {h['latency_s']:>6.1f}s")
            rows.append({
                "backend": backend, "model": model,
                "no": r["no"], "sektor": r["sektor"], "kl": r["kl"],
                "sub_indikator": r["sub_indikator"],
                "gold": r["_gold"], "pred": h.get("pred"),
                "benar": benar, "status": h["status"],
                "confidence": h.get("confidence", ""),
                "summary_pred": h.get("summary", ""),
                "source_url_pred": h.get("source_url", ""),
                "url_dari_search": h.get("url_dari_search"),
                "n_search": h["n_search"], "latency_s": h["latency_s"],
                "in_tok": h["in_tok"], "out_tok": h["out_tok"],
                "queries": " | ".join(h["queries"]),
                "error": h.get("error", ""),
                "summary_gold": r["implementation_summary"],
            })

for bk in BACKENDS:
    _save_cache(bk)

detail = pd.DataFrame(rows)
print(f"\n\nPemakaian pencarian: {SEARCH_COUNTER}")

Kombinasi: 2 model x 2 backend x 10 baris = 40 task


watsonx-qwen3-30b-a3b-in | ddg
  [ 1/10] no=1    gold=T pred=F SALAH      4x    0.3s
  [ 2/10] no=2    gold=T pred=T BENAR      4x   32.1s
  [ 3/10] no=3    gold=T pred=F SALAH      4x   62.4s
  [ 4/10] no=4    gold=T pred=F SALAH      4x   65.7s
  [ 5/10] no=5    gold=T pred=F SALAH      4x   64.4s
  [ 6/10] no=6    gold=T pred=F SALAH      4x  108.2s


h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof


  [ 7/10] no=7    gold=T pred=F SALAH      4x   90.0s


h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof


  [ 8/10] no=11   gold=F pred=F BENAR      4x  103.8s


...

h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connect

  [ 9/10] no=12   gold=F pred=F BENAR      3x   85.0s


h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof
h2 connection driver error: peer closed connection without sending TLS close_notify: https://docs.rs/rustls/latest/rustls/manual/_03_howto/index.html#unexpected-eof


  [10/10] no=44   gold=F pred=T SALAH      2x   47.2s

watsonx-qwen3-30b-a3b-in | tavily
  [ 1/10] no=1    gold=T pred=F SALAH      4x   14.8s
  [ 2/10] no=2    gold=T pred=T BENAR      4x   28.5s


UnicodeEncodeError: 'charmap' codec can't encode character '\u03c7' in position 3503: character maps to <undefined>

In [16]:
MODELS = [
    "watsonx-qwen3-30b-a3b-instruct-2507",
    "gemma-4-26B-A4B-it",
]
BACKENDS = ["tavily"]

print(f"Kombinasi: {len(MODELS)} model x {len(BACKENDS)} backend x {len(SAMPLE)} baris "
      f"= {len(MODELS)*len(BACKENDS)*len(SAMPLE)} task\n")

rows = []
for backend in BACKENDS:
    for model in MODELS:
        tag = f"{model.split('/')[-1][:24]} | {backend}"
        print(f"\n{'='*78}\n{tag}\n{'='*78}")
        for i, r in SAMPLE.iterrows():
            h = verifikasi(r, model, backend)
            ok = h["status"] == "OK"
            benar = (h.get("pred") == r["_gold"]) if ok else None
            mark = ("BENAR" if benar else "SALAH") if ok else h["status"]
            print(f"  [{i+1:2d}/{len(SAMPLE)}] no={r['no']:<4} gold={'T' if r['_gold'] else 'F'} "
                  f"pred={'T' if h.get('pred') else 'F' if ok else '-'} "
                  f"{mark:10s} {h['n_search']}x {h['latency_s']:>6.1f}s")
            rows.append({
                "backend": backend, "model": model,
                "no": r["no"], "sektor": r["sektor"], "kl": r["kl"],
                "sub_indikator": r["sub_indikator"],
                "gold": r["_gold"], "pred": h.get("pred"),
                "benar": benar, "status": h["status"],
                "confidence": h.get("confidence", ""),
                "summary_pred": h.get("summary", ""),
                "source_url_pred": h.get("source_url", ""),
                "url_dari_search": h.get("url_dari_search"),
                "n_search": h["n_search"], "latency_s": h["latency_s"],
                "in_tok": h["in_tok"], "out_tok": h["out_tok"],
                "queries": " | ".join(h["queries"]),
                "error": h.get("error", ""),
                "summary_gold": r["implementation_summary"],
            })

for bk in BACKENDS:
    _save_cache(bk)

detail = pd.DataFrame(rows)
print(f"\n\nPemakaian pencarian: {SEARCH_COUNTER}")

Kombinasi: 2 model x 1 backend x 10 baris = 20 task


watsonx-qwen3-30b-a3b-in | tavily
  [ 1/10] no=1    gold=T pred=F SALAH      4x    0.4s
  [ 2/10] no=2    gold=T pred=T BENAR      4x   18.3s
  [ 3/10] no=3    gold=T pred=F SALAH      4x   21.2s
  [ 4/10] no=4    gold=T pred=F SALAH      4x   21.0s
  [ 5/10] no=5    gold=T pred=F SALAH      4x   20.3s
  [ 6/10] no=6    gold=T pred=F SALAH      4x   16.0s
  [ 7/10] no=7    gold=T pred=F SALAH      4x   16.0s
  [ 8/10] no=11   gold=F pred=F BENAR      4x   19.2s
  [ 9/10] no=12   gold=F pred=F BENAR      4x   19.9s
  [10/10] no=44   gold=F pred=F BENAR      4x   19.3s

gemma-4-26B-A4B-it | tavily
  [ 1/10] no=1    gold=T pred=T BENAR      1x    8.9s
  [ 2/10] no=2    gold=T pred=T BENAR      2x   13.9s
  [ 3/10] no=3    gold=T pred=T BENAR      2x   14.6s
  [ 4/10] no=4    gold=T pred=T BENAR      2x   15.9s
  [ 5/10] no=5    gold=T pred=T BENAR      2x   10.0s
  [ 6/10] no=6    gold=T pred=F SALAH      3x   11.5s
  [ 7/10] no=7    g

## 8. Skor

In [17]:
def skor(g):
    ok = g[g["status"] == "OK"]
    n = len(ok)
    if n == 0:
        return pd.Series({"n_ok": 0})
    tp = ((ok["gold"]) & (ok["pred"])).sum()
    tn = ((~ok["gold"]) & (~ok["pred"].astype(bool))).sum()
    fp = ((~ok["gold"]) & (ok["pred"])).sum()
    fn = ((ok["gold"]) & (~ok["pred"].astype(bool))).sum()
    n_true, n_false = ok["gold"].sum(), (~ok["gold"]).sum()
    return pd.Series({
        "n_ok": n,
        "akurasi": round((tp + tn) / n, 3),
        "recall_TRUE": round(tp / n_true, 3) if n_true else None,
        "recall_FALSE": round(tn / n_false, 3) if n_false else None,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "url_valid": round(ok["url_dari_search"].fillna(False).mean(), 2),
        "search_per_baris": round(ok["n_search"].mean(), 1),
        "latency_avg": round(ok["latency_s"].mean(), 1),
        "tok_per_baris": int(ok[["in_tok", "out_tok"]].sum(axis=1).mean()),
    })

summary = detail.groupby(["backend", "model"]).apply(skor).reset_index()
baseline = SAMPLE["_gold"].mean()

print(f"Baseline 'asal TRUE' = {baseline:.0%} akurasi, recall_FALSE = 0%\n")
display(summary.sort_values("recall_FALSE", ascending=False))

print("\nCatatan baca:")
print("  recall_FALSE rendah -> model cenderung menjawab TRUE tanpa bukti kuat")
print("  url_valid  < 1.0    -> ada source_url yang tidak berasal dari hasil pencarian (halusinasi)")
print("  akurasi <= 0.70     -> tidak lebih baik dari menebak TRUE terus")

Baseline 'asal TRUE' = 70% akurasi, recall_FALSE = 0%



,backend,model,n_ok,akurasi,recall_TRUE,recall_FALSE,TP,TN,FP,FN,url_valid,search_per_baris,latency_avg,tok_per_baris
1,tavily,watsonx-qwen3-30b-a3b-instruct-2507,10.0,0.4,0.143,1.0,1.0,3.0,0.0,6.0,0.1,4.0,17.2,9802.0
0,tavily,gemma-4-26B-A4B-it,10.0,0.6,0.857,0.0,6.0,0.0,3.0,1.0,0.9,1.6,10.3,4187.0



Catatan baca:
  recall_FALSE rendah -> model cenderung menjawab TRUE tanpa bukti kuat
  url_valid  < 1.0    -> ada source_url yang tidak berasal dari hasil pencarian (halusinasi)
  akurasi <= 0.70     -> tidak lebih baik dari menebak TRUE terus


## 9. Simpan hasil

In [18]:
f_detail  = DATA_OUT / f"smoke_detail_{RUN_TS}.csv"
f_summary = DATA_OUT / f"smoke_summary_{RUN_TS}.csv"
detail.to_csv(f_detail, index=False)
summary.to_csv(f_summary, index=False)

print("Tersimpan:")
print(" ", f_detail)
print(" ", f_summary)

# --- proyeksi kebutuhan kredit untuk full run 292 baris ---
spb = detail[detail["status"] == "OK"]["n_search"].mean()
print(f"\nRata-rata pencarian per baris : {spb:.1f}")
print(f"Proyeksi full run 292 baris   : {int(spb*292):,} pencarian per model")
print(f"  2 model                     : {int(spb*292*2):,}")
print(f"  (Tavily gratis 1.000/bulan, Serper gratis 2.500 sekali)")

Tersimpan:
  ..\data\output\smoke_detail_20260917_131242.csv
  ..\data\output\smoke_summary_20260917_131242.csv

Rata-rata pencarian per baris : 2.8
Proyeksi full run 292 baris   : 817 pencarian per model
  2 model                     : 1,635
  (Tavily gratis 1.000/bulan, Serper gratis 2.500 sekali)


## 10. Inspeksi manual

In [19]:
salah = detail[(detail["status"] == "OK") & (~detail["benar"].astype(bool))]
print(f"{len(salah)} prediksi salah dari {len(detail[detail['status']=='OK'])} yang berhasil\n")

for _, r in salah.iterrows():
    print("=" * 90)
    print(f"no={r['no']} | {r['backend']} | {r['model'][:30]}")
    print(f"{r['sub_indikator'][:86]}")
    print(f"GOLD={r['gold']}  PRED={r['pred']}  confidence={r['confidence']}  url_asli={r['url_dari_search']}")
    print(f"\nQUERY : {r['queries'][:220]}")
    print(f"URL   : {r['source_url_pred'][:110]}")
    print(f"\nMODEL : {r['summary_pred'][:420]}")
    print(f"\nGOLD  : {str(r['summary_gold'])[:420]}")
    print()

10 prediksi salah dari 20 yang berhasil

no=1 | tavily | watsonx-qwen3-30b-a3b-instruct
KP 02.12.08 - Kabupaten/kota yang mendeklarasikan 5 Pilar STBM
GOLD=True  PRED=False  confidence=tinggi  url_asli=False

QUERY : STBM 5 pilar deklarasi kabupaten kota 2025 Kemenkes [site:kemenkes.go.id] | LAKIP Kemenkes 2025 STBM 5 pilar [site:kemenkes.go.id] | program sanitasi total 2025 Kemenkes deklarasi 5 pilar [site:bappenas.go.id] | BPS data
URL   : 

MODEL : Tidak ditemukan publikasi resmi dari Kemenkes, Bappenas, BPS, atau portal pemerintah lain yang menunjukkan pelaksanaan deklarasi 5 Pilar STBM oleh 30 kabupaten/kota pada tahun 2025. Hasil pencarian hanya menunjukkan informasi umum tentang STBM, video edukasi, atau data dari tahun sebelumnya. Tidak ada laporan kinerja, LAKIP, atau siaran pers resmi yang menyebutkan capaian indikator tersebut di tahun 2025.

GOLD  : Program berjalan, ditandai dengan penetapan alokasi anggaran daerah untuk survei verifikasi 5 pilar dan deklarasi level provin

In [20]:
# Fokus ke 3 baris FALSE - di sinilah model paling sering meleset
fokus = detail[(detail["gold"] == False) & (detail["status"] == "OK")]
for _, r in fokus.iterrows():
    tanda = "BENAR" if r["benar"] else "SALAH"
    print(f"[{tanda}] no={r['no']} | {r['backend']:6s} | {r['model'][:26]:28s} pred={r['pred']}")
    print(f"        {r['summary_pred'][:150]}")
    print()

[BENAR] no=11 | tavily | watsonx-qwen3-30b-a3b-inst   pred=False
        Tidak ditemukan publikasi resmi dari Kementerian Pertanian, BPS, atau sumber resmi lain yang menyatakan realisasi produksi vanili sebesar 1582 ton pad

[BENAR] no=12 | tavily | watsonx-qwen3-30b-a3b-inst   pred=False
        Tidak ditemukan publikasi resmi dari Kementerian Pertanian, Bappenas, BPS, atau portal pemerintah lain yang menyatakan pelaksanaan atau pencapaian ind

[BENAR] no=44 | tavily | watsonx-qwen3-30b-a3b-inst   pred=False
        Tidak ditemukan publikasi resmi dari Kementerian Sosial, Bappenas, Setkab, atau portal pemerintah lain yang menyatakan pelaksanaan atau realisasi prog

[SALAH] no=11 | tavily | gemma-4-26B-A4B-it           pred=True
        Produksi vanili pada tahun 2025 telah berjalan dan tercatat mengalami peningkatan. Berdasarkan Laporan Kinerja Ditjenbun 2025, produksi vanili tahun 2

[SALAH] no=12 | tavily | gemma-4-26B-A4B-it           pred=True
        Berdasarkan dokumen Laporan T